# DPO LLM Alignment — Colab Trainer

Train a small LLM with **Direct Preference Optimization (DPO + LoRA)** end to end,
right in your browser on a free Colab T4.

**Before you start:** set the runtime to GPU —
`Runtime → Change runtime type → T4 GPU`.

Pipeline: clone repo → install → prepare data → train → evaluate (win-rate).

## 1. Check the GPU

If this errors or shows no GPU, switch the runtime type (see above).

In [ ]:
!nvidia-smi

## 2. Clone the project

In [ ]:
!git clone https://github.com/yqx1412/rl-dpo-llm.git
%cd rl-dpo-llm

## 3. Install dependencies

Uses the pinned, known-compatible versions in `requirements.txt`.
Takes a few minutes on first run.

In [ ]:
!pip install -q -r requirements.txt

## 4. (Optional) Log in to Hugging Face

Not required for the public Qwen model + UltraFeedback dataset, but useful
if you later swap in a gated model or want to push your adapter to the Hub.

In [ ]:
# from huggingface_hub import login
# login()  # paste a token from https://huggingface.co/settings/tokens

## 5. Prepare the preference data

Downloads UltraFeedback and keeps a 2000-row subset for a fast Colab run.
Bump `--num-samples` for a stronger result if you have Colab time.

In [ ]:
!python src/prepare_data.py --num-samples 2000 --num-test 100

## 6. Train with DPO + LoRA

Reads every knob from `configs/dpo_config.yaml` (base model, LoRA rank,
`beta`, learning rate). One epoch on 2000 pairs is roughly 30–60 min on a T4.

Watch the `rewards/margins` value in the logs climb — that is DPO learning
to prefer the chosen responses.

In [ ]:
!python src/train_dpo.py --config configs/dpo_config.yaml

## 7. Evaluate — win-rate vs. the base model

Two judge options:
- **Local judge** (default below): no API key, uses a small local model.
- **OpenAI judge**: set `OPENAI_API_KEY` and pass `--judge openai` for a
  stronger judge (edit the cell).

Each pair is judged in both orders to cancel position bias.

In [ ]:
# Local judge (no key needed):
!python src/evaluate.py --config configs/dpo_config.yaml --num-eval 50 --judge local

# OpenAI judge instead (stronger). Uncomment and set your key:
# import os; os.environ['OPENAI_API_KEY'] = 'sk-...'
# !python src/evaluate.py --config configs/dpo_config.yaml --num-eval 100 --judge openai

## 8. Save your adapter (optional)

The LoRA adapter is in `outputs/dpo-adapter/`. Download it, or mount Drive
and copy it there so it survives the Colab session.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r outputs/dpo-adapter /content/drive/MyDrive/dpo-adapter

---
Once you have a win-rate number, put it in the README's resume bullets
(`50% → __%`). That number is the headline metric for this project.